# Part 2: Computer Vision Problem Formulation and CNN Prototype
## Manufacturing Defect Classification using Convolutional Neural Networks

**Dataset:** Synthetic Manufacturing Defect Image Dataset  
**Classes:** normal, scratch, dent, stain  
**Objective:** Build a CNN classifier that can identify surface defects in manufactured products


## Task 1: Problem Identification

**Selected Problem Type: Image Classification**

This dataset assigns exactly one label to each image — the type of defect (or absence of defect) visible on the product surface. There are no bounding boxes, no pixel-level masks, and no multiple-object annotations. Each image maps to a single category: `normal`, `scratch`, `dent`, or `stain`.

This is therefore a **multi-class image classification** problem. Object detection would require localizing defects within an image using bounding boxes, and semantic/instance segmentation would require pixel-wise labeling — neither of which is required or supported by this dataset structure. Classification is the correct and sufficient formulation here.

In [ ]:
# ---------- imports ----------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from PIL import Image
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

## Task 2: Dataset Exploration

In [ ]:
# ---------- load labels ----------
# assumes the dataset folder is in the same directory as this notebook
# or update DATASET_ROOT to the appropriate path
DATASET_ROOT = '.'          # change this if your dataset lives elsewhere
LABELS_CSV   = os.path.join(DATASET_ROOT, 'labels.csv')

df = pd.read_csv(LABELS_CSV)
df['filepath'] = df['filename'].apply(lambda x: os.path.join(DATASET_ROOT, x))

print('=== Basic Dataset Info ===')
print(f'Total images : {len(df)}')
print(f'Columns      : {list(df.columns)}')
print(f'Classes      : {sorted(df["class"].unique())}')
print()
print('=== Images per class ===')
class_counts = df['class'].value_counts().sort_index()
print(class_counts)
print()
print('=== Missing values ===')
print(df.isnull().sum())

In [ ]:
# ---------- class distribution bar chart ----------
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12']
bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', linewidth=0.8)
ax.set_title('Class Distribution in Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Defect Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val),
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, class_counts.max() + 20)
plt.tight_layout()
plt.savefig('results/class_distribution.png', dpi=150)
plt.show()
print('Dataset is perfectly balanced — 120 images per class.')

In [ ]:
# ---------- inspect image dimensions ----------
sample_dims = []
for fpath in df['filepath'].sample(20, random_state=42):
    if os.path.exists(fpath):
        with Image.open(fpath) as img:
            sample_dims.append(img.size + (len(img.getbands()),))

if sample_dims:
    unique_dims = Counter(sample_dims)
    print('Image dimensions (W x H x C) found in sample:')
    for dim, cnt in unique_dims.most_common():
        print(f'  {dim[0]}x{dim[1]}x{dim[2]}  —  {cnt} images')
else:
    print('Note: Image files not found at current path.')
    print('Update DATASET_ROOT to point to the downloaded dataset folder.')
    print('Expected dimensions based on dataset documentation: 128x128x3')

In [ ]:
# ---------- visualise sample images from each class ----------
CLASS_NAMES = ['dent', 'normal', 'scratch', 'stain']
SAMPLES_PER_CLASS = 4

# attempt to load real images; fall back to synthetic placeholders for illustration
def load_sample_images(df, class_name, n=4, size=(128, 128)):
    rows = df[df['class'] == class_name].sample(min(n, len(df[df['class']==class_name])), random_state=1)
    images = []
    for fpath in rows['filepath']:
        if os.path.exists(fpath):
            img = Image.open(fpath).convert('RGB').resize(size)
            images.append(np.array(img))
    return images

fig, axes = plt.subplots(len(CLASS_NAMES), SAMPLES_PER_CLASS, figsize=(12, 10))
fig.suptitle('Sample Images from Each Defect Class', fontsize=15, fontweight='bold', y=1.01)

for row_idx, cname in enumerate(CLASS_NAMES):
    imgs = load_sample_images(df, cname, n=SAMPLES_PER_CLASS)
    for col_idx in range(SAMPLES_PER_CLASS):
        ax = axes[row_idx, col_idx]
        if col_idx < len(imgs):
            ax.imshow(imgs[col_idx])
        else:
            ax.set_facecolor('#EEEEEE')
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(cname.upper(), fontsize=12, fontweight='bold', rotation=0,
                          labelpad=60, va='center')

plt.tight_layout()
plt.savefig('results/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 3: Image Preprocessing

In [ ]:
# ---------- configuration ----------
IMG_SIZE    = (128, 128)   # resize all images to this
IMG_SHAPE   = (128, 128, 3)
BATCH_SIZE  = 32
NUM_CLASSES = 4

CLASS_TO_IDX = {c: i for i, c in enumerate(sorted(df['class'].unique()))}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
print('Class index mapping:', CLASS_TO_IDX)

In [ ]:
# ---------- train / validation / test split ----------
# 70% train | 15% validation | 15% test  — stratified by class
df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df['class'], random_state=42)
df_val,   df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp['class'], random_state=42)

print(f'Training samples   : {len(df_train)}')
print(f'Validation samples : {len(df_val)}')
print(f'Test samples       : {len(df_test)}')
print()
print('Class distribution in training split:')
print(df_train['class'].value_counts().sort_index())

In [ ]:
# ---------- helper: load images into arrays ----------
def load_images_from_df(sub_df, img_size=IMG_SIZE):
    X, y = [], []
    missing = 0
    for _, row in sub_df.iterrows():
        fpath = row['filepath']
        if not os.path.exists(fpath):
            missing += 1
            continue
        img = Image.open(fpath).convert('RGB').resize(img_size)
        X.append(np.array(img, dtype='float32') / 255.0)   # normalize to [0, 1]
        y.append(CLASS_TO_IDX[row['class']])
    if missing:
        print(f'  Warning: {missing} files not found and were skipped.')
    return np.array(X), np.array(y)

print('Loading images ...')
X_train, y_train = load_images_from_df(df_train)
X_val,   y_val   = load_images_from_df(df_val)
X_test,  y_test  = load_images_from_df(df_test)

print(f'X_train shape : {X_train.shape}  — pixel range [{X_train.min():.2f}, {X_train.max():.2f}]')
print(f'X_val shape   : {X_val.shape}')
print(f'X_test shape  : {X_test.shape}')

In [ ]:
# ---------- data augmentation (training only) ----------
# augmentation is applied on-the-fly to increase effective training set diversity
datagen = ImageDataGenerator(
    rotation_range=15,          # slight rotation to mimic part orientation variation
    width_shift_range=0.10,     # horizontal shift
    height_shift_range=0.10,    # vertical shift
    horizontal_flip=True,       # defects can appear mirrored
    zoom_range=0.10,            # mild zoom
    fill_mode='nearest'
)
datagen.fit(X_train)

# one-hot encode labels for categorical cross-entropy
y_train_cat = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val_cat   = keras.utils.to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

print('Data augmentation configured.')
print('Labels one-hot encoded.')

## Task 4: CNN Model Creation

In [ ]:
# ---------- build the CNN ----------
# Architecture rationale:
#   - Three conv blocks with increasing filters (32 → 64 → 128) to learn
#     low-, mid-, and high-level features progressively.
#   - Batch normalisation after each conv to stabilise training.
#   - MaxPooling to reduce spatial dimensions and introduce translation invariance.
#   - Dropout before the dense layers to regularise and reduce overfitting.

def build_cnn(input_shape=IMG_SHAPE, num_classes=NUM_CLASSES):
    model = models.Sequential([
        # --- input ---
        layers.Input(shape=input_shape),

        # --- Block 1: detect basic edges and textures ---
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),          # 128 → 64
        layers.Dropout(0.25),

        # --- Block 2: detect shapes and patterns ---
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),          # 64 → 32
        layers.Dropout(0.25),

        # --- Block 3: detect high-level defect features ---
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),          # 32 → 16
        layers.Dropout(0.25),

        # --- classifier head ---
        layers.Flatten(),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.50),
        layers.Dense(num_classes, activation='softmax')  # output: probability per class
    ], name='defect_cnn')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_cnn()
model.summary()

## Task 5: Model Training and Evaluation

In [ ]:
# ---------- callbacks ----------
early_stop = EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1
)

# ---------- train ----------
EPOCHS = 40

history = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE, seed=42),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# ---------- accuracy / loss curves ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
epochs_ran = range(1, len(history.history['accuracy']) + 1)

ax1.plot(epochs_ran, history.history['accuracy'],     label='Train', color='steelblue',  linewidth=2)
ax1.plot(epochs_ran, history.history['val_accuracy'], label='Validation', color='coral', linewidth=2, linestyle='--')
ax1.set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_ran, history.history['loss'],     label='Train', color='steelblue',  linewidth=2)
ax2.plot(epochs_ran, history.history['val_loss'], label='Validation', color='coral', linewidth=2, linestyle='--')
ax2.set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Model Training Curves — Manufacturing Defect CNN', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/accuracy_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---------- test set evaluation ----------
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

y_pred_prob = model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

print()
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=sorted(CLASS_TO_IDX.keys())))

In [ ]:
# ---------- confusion matrix ----------
cm = confusion_matrix(y_test, y_pred)
class_labels = sorted(CLASS_TO_IDX.keys())

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=150)
plt.show()

# brief interpretation
print()
print('Interpretation:')
print('  - Diagonal cells show correctly classified images.')
print('  - Off-diagonal cells indicate misclassifications.')
print('  - A near-diagonal matrix confirms the model has learned class-specific features.')

In [ ]:
# ---------- sample predictions on test images ----------
n_display = 12
indices   = np.random.choice(len(X_test), n_display, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for i, idx in enumerate(indices):
    ax = axes[i]
    ax.imshow(X_test[idx])
    true_label = IDX_TO_CLASS[y_test[idx]]
    pred_label = IDX_TO_CLASS[y_pred[idx]]
    conf       = y_pred_prob[idx][y_pred[idx]] * 100
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({conf:.1f}%)',
                 color=color, fontsize=9, fontweight='bold')
    ax.axis('off')

fig.suptitle('Sample Predictions on Test Images\n(green = correct, red = incorrect)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_predictions/prediction_outputs.png', dpi=150, bbox_inches='tight')
plt.show()

## Task 6: CNN Concept Explanation

### What is Convolution?
Convolution is a mathematical operation that slides a small learnable filter (kernel) across an image and computes a dot product at each position. This produces a **feature map** that highlights where a particular pattern (edge, corner, texture) exists in the input. Unlike a dense layer that treats every pixel independently, a convolutional layer shares weights across spatial positions, which drastically reduces parameters and preserves spatial relationships.

### Why is Pooling Used?
After convolution, feature maps are typically large. **Max pooling** takes the maximum value in a local neighbourhood (say 2×2), reducing the spatial size by half. This achieves two things: it reduces computation for subsequent layers, and it provides a form of **translation invariance** — a defect that shifts by a pixel or two still produces the same pooled output. This robustness is important in manufacturing inspection where the camera angle or product position can vary slightly.

### Why is ReLU Commonly Used in CNNs?
ReLU (Rectified Linear Unit) outputs `max(0, x)`, which is computationally cheap and avoids the **vanishing gradient problem** that plagues sigmoid and tanh for deep networks. Because ReLU gradients are either 0 or 1 (not squashed), gradients flow cleanly through many layers during backpropagation, enabling effective training of deep architectures. It also introduces sparsity — many neurons output zero — which makes representations more interpretable.

### Why are CNNs Better than Feed-Forward Networks for Image Data?
A standard feed-forward (dense) network treats each pixel as an independent input. A 128×128×3 image has 49,152 inputs; connecting all of them to even a modest hidden layer creates millions of parameters, leading to overfitting and enormous computational cost. CNNs exploit three key properties of images:
1. **Local connectivity** — nearby pixels are more correlated than distant ones.
2. **Weight sharing** — the same filter detects the same feature (e.g., an edge) anywhere in the image.
3. **Hierarchical representation** — early layers detect edges, middle layers detect shapes, deep layers detect class-specific patterns (like the circular imprint of a dent).
These inductive biases give CNNs strong generalisation with far fewer parameters than an equivalent dense network.


## Task 7: Business Use Case Mapping

### Application Domain: Manufacturing / Quality Inspection

In a real production line, products move past a camera at high speed. A human inspector can only examine a fraction of items, is subject to fatigue, and can be inconsistent across shifts. A CNN-based defect classifier addresses all three limitations.

**Concrete implementation scenario:**  
A consumer electronics manufacturer mounts a 5 MP line-scan camera above the conveyor belt. Each captured image is resized to 128×128, normalised, and fed to the trained CNN in under 10 ms. The model outputs one of four labels. Units classified as `scratch`, `dent`, or `stain` are diverted to a secondary inspection bay; units labelled `normal` proceed to packaging.

**Business benefits:**
- **Consistency:** The model applies the same decision boundary 24 hours a day with no fatigue.
- **Speed:** Inference at ~100 fps on a basic GPU processes an entire production shift in real time.
- **Traceability:** Every defect prediction is logged with a timestamp and image, enabling process engineers to trace quality issues back to specific machine settings or raw material batches.
- **Cost reduction:** Automated rejection of defective units before packaging reduces return rates and warranty claims.

**Extensibility:**  
With more labelled data, the same architecture can be extended to localize defect regions (object detection) or estimate defect severity (regression), supporting a fully automated quality-grading pipeline.


In [ ]:
# ---------- save model ----------
model.save('results/defect_cnn_model.keras')
print('Model saved to results/defect_cnn_model.keras')

In [ ]:
# ---------- final summary ----------
print('=' * 50)
print('         FINAL RESULTS SUMMARY')
print('=' * 50)
print(f'  Architecture    : Custom 3-block CNN')
print(f'  Input shape     : {IMG_SHAPE}')
print(f'  Classes         : {NUM_CLASSES} ({list(CLASS_TO_IDX.keys())})')
print(f'  Training images : {len(X_train)}')
print(f'  Test images     : {len(X_test)}')
print(f'  Test Accuracy   : {test_acc*100:.2f}%')
print(f'  Test Loss       : {test_loss:.4f}')
print('=' * 50)